In [0]:
# ── Silver Layer Configuration ─────────────────
base_path    = "abfss://sapdata@adlsloki.dfs.core.windows.net"
bronze_path  = f"{base_path}/Bronze"
silver_path  = f"{base_path}/Silver"

spark.sql("USE CATALOG sap_catalog")
print("✅ Config ready!")

In [0]:
# ── Read Bronze Delta Tables ───────────────────
vbak = spark.table("sap_catalog.bronze_schema.vbak_raw")
vbap = spark.table("sap_catalog.bronze_schema.vbap_raw")
kna1 = spark.table("sap_catalog.bronze_schema.kna1_raw")
mara = spark.table("sap_catalog.bronze_schema.mara_raw")
bkpf = spark.table("sap_catalog.bronze_schema.bkpf_raw")
bseg = spark.table("sap_catalog.bronze_schema.bseg_raw")

print("✅ Bronze tables loaded!")
print(f"VBAK rows: {vbak.count():,}")
print(f"VBAP rows: {vbap.count():,}")
print(f"KNA1 rows: {kna1.count():,}")
print(f"MARA rows: {mara.count():,}")
print(f"BKPF rows: {bkpf.count():,}")
print(f"BSEG rows: {bseg.count():,}")

In [0]:
# ── Check All Bronze Table Columns ─────────────

print("="*50)
print("VBAK Columns (Sales Order Header):")
print("="*50)
for col_name in vbak.columns:
    print(f"  {col_name}")

print("\n" + "="*50)
print("VBAP Columns (Sales Order Items):")
print("="*50)
for col_name in vbap.columns:
    print(f"  {col_name}")

print("\n" + "="*50)
print("KNA1 Columns (Customer Master):")
print("="*50)
for col_name in kna1.columns:
    print(f"  {col_name}")

print("\n" + "="*50)
print("MARA Columns (Material Master):")
print("="*50)
for col_name in mara.columns:
    print(f"  {col_name}")

print("\n" + "="*50)
print("BKPF Columns (Accounting Header):")
print("="*50)
for col_name in bkpf.columns:
    print(f"  {col_name}")

print("\n" + "="*50)
print("BSEG Columns (Accounting Items):")
print("="*50)
for col_name in bseg.columns:
    print(f"  {col_name}")

In [0]:
from pyspark.sql.functions import (
    col, trim, upper, when, 
    lit, current_timestamp, regexp_replace
)

def transform_vbak(df):
    return (df
        .select(
            col("mandt").alias("client"),
            col("vbeln").alias("sales_order_number"),
            col("posnr").alias("line_item_number"),
            col("matnr").alias("material_number"),
            col("arktx").alias("item_description"),
            col("netwr").cast("double").alias("net_value"),
            col("waerk").alias("currency"),
            col("kwmeng").cast("double").alias("order_quantity"),
            col("meins").alias("unit_of_measure"),
            col("netpr").cast("double").alias("net_price"),
            col("werks").alias("plant"),
            col("spart").alias("division"),
            col("erdat").alias("created_date"),
            col("ernam").alias("created_by"),
            col("_ingestion_ts"),
            col("_source_system")
        )
        .withColumn("sales_order_number",
            regexp_replace(col("sales_order_number"), "^0+", ""))
        .withColumn("material_number",
            regexp_replace(col("material_number"), "^0+", ""))
        .withColumn("net_value",
            when(col("net_value").isNull(), lit(0.0))
            .otherwise(col("net_value")))
        .withColumn("order_quantity",
            when(col("order_quantity").isNull(), lit(0.0))
            .otherwise(col("order_quantity")))
        .withColumn("line_total",
            col("order_quantity") * col("net_price"))
        .withColumn("_silver_ts", current_timestamp())
        .withColumn("_layer", lit("silver"))
        .dropDuplicates(["sales_order_number", "line_item_number"])
        .filter(col("sales_order_number").isNotNull())
    )

vbak_silver = transform_vbak(vbak)
print(f"✅ VBAK Silver: {vbak_silver.count():,} rows")
display(vbak_silver.limit(3))

In [0]:
def transform_kna1(df):
    return (df
        .select(
            col("mandt").alias("client"),
            col("kunnr").alias("customer_id"),
            col("name1").alias("customer_name"),
            col("name2").alias("customer_name2"),
            col("ort01").alias("city"),
            col("land1").alias("country"),
            col("pstlz").alias("postal_code"),
            col("regio").alias("region"),
            col("stras").alias("street"),
            col("telf1").alias("phone"),
            col("telfx").alias("fax"),
            col("brsch").alias("industry"),
            col("erdat").alias("created_date"),
            col("ernam").alias("created_by"),
            col("_ingestion_ts"),
            col("_source_system")
        )
        .withColumn("customer_id",
            regexp_replace(col("customer_id"), "^0+", ""))
        .withColumn("customer_name",
            trim(col("customer_name")))
        .withColumn("country",
            upper(col("country")))
        .withColumn("_silver_ts", current_timestamp())
        .withColumn("_layer", lit("silver"))
        .dropDuplicates(["customer_id"])
        .filter(col("customer_id").isNotNull())
        .filter(col("customer_name").isNotNull())
    )

kna1_silver = transform_kna1(kna1)
print(f"✅ KNA1 Silver: {kna1_silver.count():,} rows")
display(kna1_silver.limit(3))

In [0]:
def transform_mara(df):
    return (df
        .select(
            col("mandt").alias("client"),
            col("matnr").alias("material_number"),
            col("mtart").alias("material_type"),
            col("matkl").alias("material_group"),
            col("meins").alias("base_unit"),
            col("brgew").cast("double").alias("gross_weight"),
            col("ntgew").cast("double").alias("net_weight"),
            col("gewei").alias("weight_unit"),
            col("volum").cast("double").alias("volume"),
            col("voleh").alias("volume_unit"),
            col("spart").alias("division"),
            col("labor").alias("laboratory"),
            col("ersda").alias("created_date"),
            col("ernam").alias("created_by"),
            col("_ingestion_ts"),
            col("_source_system")
        )
        .withColumn("material_number",
            regexp_replace(col("material_number"), "^0+", ""))
        .withColumn("gross_weight",
            when(col("gross_weight").isNull(), lit(0.0))
            .otherwise(col("gross_weight")))
        .withColumn("net_weight",
            when(col("net_weight").isNull(), lit(0.0))
            .otherwise(col("net_weight")))
        .withColumn("_silver_ts", current_timestamp())
        .withColumn("_layer", lit("silver"))
        .dropDuplicates(["material_number"])
        .filter(col("material_number").isNotNull())
    )

mara_silver = transform_mara(mara)
print(f"✅ MARA Silver: {mara_silver.count():,} rows")
display(mara_silver.limit(3))

In [0]:
def transform_bkpf(df):
    return (df
        .select(
            col("mandt").alias("client"),
            col("bukrs").alias("company_code"),
            col("belnr").alias("document_number"),
            col("gjahr").alias("fiscal_year"),
            col("blart").alias("document_type"),
            col("bldat").alias("document_date"),
            col("budat").alias("posting_date"),
            col("monat").alias("fiscal_period"),
            col("waers").alias("currency"),
            col("kursf").cast("double").alias("exchange_rate"),
            col("bktxt").alias("document_text"),
            col("usnam").alias("created_by"),
            col("tcode").alias("transaction_code"),
            col("bstat").alias("document_status"),
            col("_ingestion_ts"),
            col("_source_system")
        )
        .withColumn("document_number",
            regexp_replace(col("document_number"), "^0+", ""))
        .withColumn("_silver_ts", current_timestamp())
        .withColumn("_layer", lit("silver"))
        .dropDuplicates(["company_code","document_number","fiscal_year"])
        .filter(col("document_number").isNotNull())
        .filter(col("company_code").isNotNull())
    )

bkpf_silver = transform_bkpf(bkpf)
print(f"✅ BKPF Silver: {bkpf_silver.count():,} rows")
display(bkpf_silver.limit(3))

In [0]:
def transform_bseg(df):
    return (df
        .select(
            col("mandt").alias("client"),
            col("bukrs").alias("company_code"),
            col("belnr").alias("document_number"),
            col("gjahr").alias("fiscal_year"),
            col("buzei").alias("line_item"),
            col("koart").alias("account_type"),
            col("shkzg").alias("debit_credit"),
            col("dmbtr").cast("double").alias("amount_local"),
            col("wrbtr").cast("double").alias("amount_foreign"),
            col("hkont").alias("gl_account"),
            col("kunnr").alias("customer_id"),
            col("lifnr").alias("vendor_id"),
            col("kostl").alias("cost_center"),
            col("matnr").alias("material_number"),
            col("menge").cast("double").alias("quantity"),
            col("meins").alias("unit"),
            col("sgtxt").alias("line_text"),
            col("zuonr").alias("assignment"),
            col("_ingestion_ts"),
            col("_source_system")
        )
        .withColumn("document_number",
            regexp_replace(col("document_number"), "^0+", ""))
        .withColumn("customer_id",
            when(col("customer_id").isNotNull(),
                regexp_replace(col("customer_id"), "^0+", ""))
            .otherwise(lit(None)))
        .withColumn("amount_local",
            when(col("amount_local").isNull(), lit(0.0))
            .otherwise(col("amount_local")))
        .withColumn("_silver_ts", current_timestamp())
        .withColumn("_layer", lit("silver"))
        .filter(col("document_number").isNotNull())
    )

bseg_silver = transform_bseg(bseg)
print(f"✅ BSEG Silver: {bseg_silver.count():,} rows")
display(bseg_silver.limit(3))

In [0]:
# ── Write to Silver Schema ──────────────────────
def write_silver(df, table_name):
    target = f"sap_catalog.silver_schema.{table_name}"
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(target))
    print(f"✅ Written: {target} — {df.count():,} rows")

write_silver(vbak_silver, "vbak_silver")
write_silver(kna1_silver, "kna1_silver")
write_silver(mara_silver, "mara_silver")
write_silver(bkpf_silver, "bkpf_silver")
write_silver(bseg_silver, "bseg_silver")

print("\n" + "="*40)
print("SILVER LAYER COMPLETE!")
display(spark.sql("SHOW TABLES IN sap_catalog.silver_schema"))